# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and analyze the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display dataset title and summary
print(f"{metadata.name}: {metadata.description}")
print("\nPublished:", metadata.datePublished)
print("\nKeywords:", metadata.keywords)
print("\nLicense:", metadata.license)
print("\nSpatial Coverage:", metadata.spatialCoverage)
print("\nTemporal Coverage:", metadata.temporalCoverage)

## 2. Data Overview
Review available record sets, their fields, and their unique `@id`s. This helps identify data structure and enables referencing entities by `@id`.

In [ ]:
# List available record sets and their fields, referencing each by their `@id`
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record sets.")
overview = {}
for rs in record_sets:
    print(f"\nRecordSet: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    print(f"  Description: {rs.get('description', 'N/A')}")

    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("  Fields:")
    for f in fields:
        field_id = f['@id'] if isinstance(f, dict) and '@id' in f else f
        print(f"    - {field_id}")
        overview.setdefault(rs['@id'], []).append(field_id)

    columns = rs.get('column', [])
    if columns:
        print("  Columns:")
        for col in columns:
            col_id = col['@id'] if isinstance(col, dict) and '@id' in col else col
            print(f"    - {col_id}")

## 3. Data Extraction
Load data from each record set (using their `@id`) into pandas DataFrames. Use the overview above to choose record sets and fields for analysis.

In [ ]:
# Prepare to read every record set using its `@id`
dataframes = {}
for rs in record_sets:
    rs_id = rs['@id']
    try:
        records_iter = dataset.records(record_set=rs_id)
        records = list(records_iter)
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"\nRecord set '{rs_id}' loaded with shape {df.shape}.")
            print("Columns:", df.columns.tolist())
            print(df.head())
        else:
            print(f"\nRecord set '{rs_id}': No records found.")
    except Exception as e:
        print(f"\nRecord set '{rs_id}': Failed to load. Error: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply standard data processing steps—filter records by criteria, normalize numeric fields, handle missing data, and group records by key attributes. Use `@id` references for all fields.

In [ ]:
# Choose a data table for EDA — use the `@id` from overview (adjust below as needed)
if dataframes:
    # For demonstration, pick the first loaded RecordSet
    selected_rs_id = list(dataframes.keys())[0]
    df = dataframes[selected_rs_id]

    # Identify numeric fields by inspecting dtypes
    numeric_fields = df.select_dtypes(include='number').columns.tolist()
    print(f"Numeric fields in record set {selected_rs_id}: {numeric_fields}")

    # Select a numeric field for filtering and normalization
    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # reference by column `@id`
        threshold = df[numeric_field_id].mean()  # example threshold: mean
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with '{numeric_field_id}' > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize selected numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Identify a field suitable for grouping (categorical)
        candidate_group_fields = [c for c in df.columns if df[c].dtype == 'object' and c != numeric_field_id]
        if candidate_group_fields:
            group_field_id = candidate_group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean '{numeric_field_id}' by '{group_field_id}':")
            print(grouped_df.head())
        else:
            print("No categorical group fields found.")
    else:
        print("No numeric fields to process.")
else:
    print("No record sets loaded for EDA.")

## 5. Visualization
Visualize field distributions or relationships in the dataset. All visualizations reference fields and record sets by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of a numeric field (referenced by @id)
if dataframes and numeric_fields:
    # Histogram
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' in '{selected_rs_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot grouped by a categorical field
    if candidate_group_fields:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=df[candidate_group_fields[0]], y=df[numeric_field_id])
        plt.title(f"'{numeric_field_id}' by '{candidate_group_fields[0]}'")
        plt.xlabel(candidate_group_fields[0])
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Unable to visualize: no data or numeric fields found.")

## 6. Conclusion
In this notebook, we've loaded the FAIR^2 dataset using `mlcroissant`, reviewed its metadata and structure, extracted record sets and fields by their `@id`, performed filtering and normalization, grouped records, and visualized distributions. By referencing all entities via their `@id`, we've preserved the dataset structure for reproducible FAIR workflows. For full analysis, consult the [dataset documentation](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) and adapt the notebook for specific research needs.